In [8]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import pandas as pd
import numpy as np
from scipy.stats import randint
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix
from sklearn import metrics
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
df = pd.read_excel('train.xlsx')
df.shape

(6853, 4)

In [3]:
df1 = df[['content','concerns']].copy()

In [4]:
df1

,content,concerns
0,@imaerell @BradfatherSpeak @ALEXNEWMAN_JOU New...,side-effect unnecessary
1,"@theousherwood @LBC I’m not anti vaccine, but ...",pharma
2,@BorisJohnson I won’t be taking any vaccine ev...,none
3,@LPerrins They have set this up that nothing w...,ineffective
4,@AngelaDeAngelo I believe I read it on one of ...,rushed
...,...,...
6848,We do not need another unknown illness to be m...,side-effect
6849,"@ZagamaMas I would be wary of AstraZenica's ""O...",rushed
6850,@GeoohhM Lol I mean it really all just depends...,unnecessary
6851,@CanAditude Seriously don't care what she has ...,mandatory


In [5]:
df1 = df1[pd.notnull(df1['content'])]
df1

,content,concerns
0,@imaerell @BradfatherSpeak @ALEXNEWMAN_JOU New...,side-effect unnecessary
1,"@theousherwood @LBC I’m not anti vaccine, but ...",pharma
2,@BorisJohnson I won’t be taking any vaccine ev...,none
3,@LPerrins They have set this up that nothing w...,ineffective
4,@AngelaDeAngelo I believe I read it on one of ...,rushed
...,...,...
6848,We do not need another unknown illness to be m...,side-effect
6849,"@ZagamaMas I would be wary of AstraZenica's ""O...",rushed
6850,@GeoohhM Lol I mean it really all just depends...,unnecessary
6851,@CanAditude Seriously don't care what she has ...,mandatory


In [6]:
pd.DataFrame(df1.concerns.unique()).values

array([['side-effect unnecessary'],
       ['pharma'],
       ['none'],
       ['ineffective'],
       ['rushed'],
       ['political'],
       ['side-effect'],
       ['conspiracy side-effect'],
       ['political rushed'],
       ['rushed ineffective'],
       ['mandatory ineffective'],
       ['ingredients'],
       ['pharma political'],
       ['unnecessary side-effect'],
       ['rushed side-effect'],
       ['conspiracy'],
       ['unnecessary'],
       ['political side-effect'],
       ['side-effect rushed'],
       ['side-effect political'],
       ['conspiracy ingredients'],
       ['unnecessary pharma'],
       ['mandatory'],
       ['rushed unnecessary'],
       ['conspiracy pharma'],
       ['side-effect pharma'],
       ['rushed pharma'],
       ['rushed political'],
       ['side-effect ineffective'],
       ['ineffective side-effect'],
       ['unnecessary mandatory'],
       ['side-effect ingredients'],
       ['conspiracy rushed side-effect'],
       ['rushed ingredien

In [7]:
X = df1['content']
y = [concerns.split() for concerns in df1['concerns']]

In [9]:
tfidf_vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=4, ngram_range=(1, 2), stop_words='english')
X_tfidf = tfidf_vectorizer.fit_transform(X)

In [10]:
mlb = MultiLabelBinarizer()
y_binary = mlb.fit_transform(y)


## PreProcessing

In [12]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
def decontract(sentence):
    sentence = re.sub(r"n\'t", " not", sentence)
    sentence = re.sub(r"\'re", " are", sentence)
    sentence = re.sub(r"\'s", " is", sentence)
    sentence = re.sub(r"\'d", " would", sentence)
    sentence = re.sub(r"\'ll", " will", sentence)
    sentence = re.sub(r"\'t", " not", sentence)
    sentence = re.sub(r"\'ve", " have", sentence)
    sentence = re.sub(r"\'m", " am", sentence)
    return sentence

def removePunctuation(sentence): 
    sentence = re.sub(r'[?|!|\'|"|#]',r'',sentence)
    sentence = re.sub(r'[@.|,|)|(|\|/]',r' ',sentence)
    sentence = sentence.strip()
    sentence = sentence.replace("\n"," ")
    return sentence

def removeNumber(sentence):
    alpha_sent = ""
    for word in sentence.split():
        alpha_word = re.sub('[^a-z A-Z]+', '', word)
        alpha_sent += alpha_word
        alpha_sent += " "
    alpha_sent = alpha_sent.strip()
    return alpha_sent

def removeStopWords(sentence):
    return stopwords.sub("", sentence)
def stemming(sentence):
    stemmer = SnowballStemmer("english")
    stemmedSentence = ""
    for word in sentence.split():
        stem = stemmer.stem(word)
        stemmedSentence += stem
        stemmedSentence += " "
    stemmedSentence = stemmedSentence.strip()
    return stemmedSentence

## Trying LinearSVC

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

X_train, X_test, y_train, y_test = train_test_split(df1['content'], df1['concerns'], test_size=0.2, random_state=42)
tfidf_vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=4, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train.str.split())
y_test_encoded = mlb.transform(y_test.str.split())
classifier = LinearSVC()
classifier.fit(X_train_tfidf, y_train_encoded.argmax(axis=1))
y_pred = classifier.predict(X_test_tfidf)
print(classification_report(y_test_encoded.argmax(axis=1), y_pred, target_names=mlb.classes_))

/home/aayush/.local/lib/python3.10/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


              precision    recall  f1-score   support

  conspiracy       0.46      0.43      0.44        61
     country       0.45      0.38      0.41        24
 ineffective       0.60      0.62      0.61       232
 ingredients       0.64      0.30      0.41        47
   mandatory       0.60      0.56      0.58        89
        none       0.48      0.32      0.38        95
      pharma       0.46      0.53      0.50       141
   political       0.42      0.30      0.35        61
   religious       0.33      0.17      0.22         6
      rushed       0.52      0.47      0.50       159
 side-effect       0.63      0.78      0.70       405
 unnecessary       0.38      0.29      0.33        51

    accuracy                           0.56      1371
   macro avg       0.50      0.43      0.45      1371
weighted avg       0.55      0.56      0.55      1371



## Linear SVC, RandomForest, MultinomialNB, Logistic Regression

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer
X_train, X_test, y_train, y_test = train_test_split(df['content'], df['concerns'], test_size=0.2, random_state=42)
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train.str.split())
y_test_encoded = mlb.transform(y_test.str.split())
classifiers = {
    'LinearSVC': LinearSVC(),
    'RandomForest': RandomForestClassifier(),
    'MultinomialNB': MultinomialNB(),  # Adjusted for Multinomial Naive Bayes
    'LogisticRegression': LogisticRegression()
}
for name, classifier in classifiers.items():
    print(f"\nTraining and evaluating {name}...")
    if name == 'MultinomialNB':
        X_train_nb = X_train_tfidf.astype(float)
        X_test_nb = X_test_tfidf.astype(float)
        classifier.fit(X_train_nb, y_train_encoded.argmax(axis=1))
    else:
        classifier.fit(X_train_tfidf, y_train_encoded.argmax(axis=1))
    
    y_pred = classifier.predict(X_test_tfidf)
    
    print(f"\nResults for {name}:")
    print(classification_report(y_test_encoded.argmax(axis=1), y_pred, target_names=mlb.classes_))


Training and evaluating LinearSVC...

Results for LinearSVC:


/home/aayush/.local/lib/python3.10/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


              precision    recall  f1-score   support

  conspiracy       0.50      0.43      0.46        61
     country       0.50      0.38      0.43        24
 ineffective       0.57      0.58      0.57       232
 ingredients       0.58      0.32      0.41        47
   mandatory       0.62      0.55      0.58        89
        none       0.46      0.35      0.40        95
      pharma       0.45      0.52      0.49       141
   political       0.42      0.31      0.36        61
   religious       0.50      0.17      0.25         6
      rushed       0.54      0.50      0.52       159
 side-effect       0.63      0.78      0.70       405
 unnecessary       0.38      0.25      0.31        51

    accuracy                           0.56      1371
   macro avg       0.51      0.43      0.46      1371
weighted avg       0.55      0.56      0.55      1371


Training and evaluating RandomForest...

Results for RandomForest:
              precision    recall  f1-score   support

  conspira

/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for LogisticRegression:
              precision    recall  f1-score   support

  conspiracy       0.65      0.25      0.36        61
     country       0.67      0.17      0.27        24
 ineffective       0.57      0.65      0.61       232
 ingredients       0.91      0.21      0.34        47
   mandatory       0.74      0.48      0.59        89
        none       0.51      0.20      0.29        95
      pharma       0.51      0.56      0.54       141
   political       0.55      0.10      0.17        61
   religious       0.00      0.00      0.00         6
      rushed       0.58      0.43      0.49       159
 side-effect       0.52      0.86      0.65       405
 unnecessary       0.76      0.25      0.38        51

    accuracy                           0.55      1371
   macro avg       0.58      0.35      0.39      1371
weighted avg       0.58      0.55      0.52      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and bei

## Logistic Regression for different parameters

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

X_train, X_test, y_train, y_test = train_test_split(df['content'], df['concerns'], test_size=0.2, random_state=42)
tfidf_vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=4, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train.str.split())
y_test_encoded = mlb.transform(y_test.str.split())

for c_value in [0.001, 0.01, 0.1, 1, 10, 100]:
    classifier = LogisticRegression(C=c_value, max_iter=1000) 
    classifier.fit(X_train_tfidf, y_train_encoded.argmax(axis=1))
    y_pred = classifier.predict(X_test_tfidf)
    print(f"\nResults for Logistic Regression with C={c_value}:")
    print(classification_report(y_test_encoded.argmax(axis=1), y_pred, target_names=mlb.classes_))


Results for Logistic Regression with C=0.001:
              precision    recall  f1-score   support

  conspiracy       0.00      0.00      0.00        61
     country       0.00      0.00      0.00        24
 ineffective       0.00      0.00      0.00       232
 ingredients       0.00      0.00      0.00        47
   mandatory       0.00      0.00      0.00        89
        none       0.00      0.00      0.00        95
      pharma       0.00      0.00      0.00       141
   political       0.00      0.00      0.00        61
   religious       0.00      0.00      0.00         6
      rushed       0.00      0.00      0.00       159
 side-effect       0.30      1.00      0.46       405
 unnecessary       0.00      0.00      0.00        51

    accuracy                           0.30      1371
   macro avg       0.02      0.08      0.04      1371
weighted avg       0.09      0.30      0.13      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=0.01:
              precision    recall  f1-score   support

  conspiracy       0.00      0.00      0.00        61
     country       0.00      0.00      0.00        24
 ineffective       0.00      0.00      0.00       232
 ingredients       0.00      0.00      0.00        47
   mandatory       0.00      0.00      0.00        89
        none       0.00      0.00      0.00        95
      pharma       0.00      0.00      0.00       141
   political       0.00      0.00      0.00        61
   religious       0.00      0.00      0.00         6
      rushed       0.00      0.00      0.00       159
 side-effect       0.30      1.00      0.46       405
 unnecessary       0.00      0.00      0.00        51

    accuracy                           0.30      1371
   macro avg       0.02      0.08      0.04      1371
weighted avg       0.09      0.30      0.13      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=0.1:
              precision    recall  f1-score   support

  conspiracy       0.00      0.00      0.00        61
     country       0.00      0.00      0.00        24
 ineffective       0.71      0.34      0.46       232
 ingredients       0.00      0.00      0.00        47
   mandatory       0.00      0.00      0.00        89
        none       0.00      0.00      0.00        95
      pharma       0.62      0.06      0.10       141
   political       0.00      0.00      0.00        61
   religious       0.00      0.00      0.00         6
      rushed       0.71      0.03      0.06       159
 side-effect       0.32      0.98      0.48       405
 unnecessary       0.00      0.00      0.00        51

    accuracy                           0.36      1371
   macro avg       0.20      0.12      0.09      1371
weighted avg       0.36      0.36      0.24      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=1:
              precision    recall  f1-score   support

  conspiracy       0.65      0.25      0.36        61
     country       0.58      0.29      0.39        24
 ineffective       0.57      0.66      0.61       232
 ingredients       1.00      0.17      0.29        47
   mandatory       0.71      0.47      0.57        89
        none       0.57      0.17      0.26        95
      pharma       0.53      0.55      0.54       141
   political       0.50      0.07      0.12        61
   religious       0.00      0.00      0.00         6
      rushed       0.57      0.43      0.49       159
 side-effect       0.52      0.87      0.65       405
 unnecessary       0.67      0.27      0.39        51

    accuracy                           0.55      1371
   macro avg       0.57      0.35      0.39      1371
weighted avg       0.58      0.55      0.51      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=10:
              precision    recall  f1-score   support

  conspiracy       0.52      0.43      0.47        61
     country       0.56      0.38      0.45        24
 ineffective       0.60      0.64      0.62       232
 ingredients       0.72      0.28      0.40        47
   mandatory       0.62      0.54      0.58        89
        none       0.47      0.33      0.39        95
      pharma       0.49      0.57      0.53       141
   political       0.46      0.30      0.36        61
   religious       1.00      0.17      0.29         6
      rushed       0.55      0.47      0.51       159
 side-effect       0.61      0.79      0.69       405
 unnecessary       0.43      0.29      0.35        51

    accuracy                           0.57      1371
   macro avg       0.59      0.43      0.47      1371
weighted avg       0.57      0.57      0.56      1371


Results for Logistic Regression with C=100:
              precision    recall  f1-score 

In [17]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

import nltk
nltk.download('stopwords')
nltk.download('punkt')

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'[^A-Za-z\s]', '', text)
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]
    preprocessed_text = ' '.join(tokens)

    return preprocessed_text
df['content_preprocessed'] = df['content'].apply(preprocess_text)
X_train, X_test, y_train, y_test = train_test_split(df['content_preprocessed'], df['concerns'], test_size=0.2, random_state=42)
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train.str.split())
y_test_encoded = mlb.transform(y_test.str.split())
for c_value in [1,2,3,4,5,6,7,8,9,10]:
    classifier = LogisticRegression(C=c_value, max_iter=1000)
    classifier.fit(X_train_tfidf, y_train_encoded.argmax(axis=1))
    y_pred = classifier.predict(X_test_tfidf)
    print(f"\nResults for Logistic Regression with C={c_value}:")
    print(classification_report(y_test_encoded.argmax(axis=1), y_pred, target_names=mlb.classes_))

[nltk_data] Downloading package stopwords to /home/aayush/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/aayush/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Results for Logistic Regression with C=1:
              precision    recall  f1-score   support

  conspiracy       0.68      0.25      0.36        61
     country       0.67      0.17      0.27        24
 ineffective       0.54      0.67      0.60       232
 ingredients       0.92      0.23      0.37        47
   mandatory       0.74      0.51      0.60        89
        none       0.56      0.21      0.31        95
      pharma       0.51      0.63      0.56       141
   political       0.83      0.25      0.38        61
   religious       0.00      0.00      0.00         6
      rushed       0.65      0.50      0.56       159
 side-effect       0.57      0.85      0.68       405
 unnecessary       0.67      0.31      0.43        51

    accuracy                           0.58      1371
   macro avg       0.61      0.38      0.43      1371
weighted avg       0.61      0.58      0.55      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=2:
              precision    recall  f1-score   support

  conspiracy       0.59      0.33      0.42        61
     country       0.70      0.29      0.41        24
 ineffective       0.54      0.66      0.59       232
 ingredients       0.88      0.32      0.47        47
   mandatory       0.66      0.55      0.60        89
        none       0.46      0.24      0.32        95
      pharma       0.51      0.65      0.57       141
   political       0.67      0.33      0.44        61
   religious       0.00      0.00      0.00         6
      rushed       0.62      0.48      0.54       159
 side-effect       0.61      0.82      0.70       405
 unnecessary       0.59      0.31      0.41        51

    accuracy                           0.59      1371
   macro avg       0.57      0.42      0.46      1371
weighted avg       0.59      0.59      0.57      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=3:
              precision    recall  f1-score   support

  conspiracy       0.53      0.31      0.39        61
     country       0.73      0.33      0.46        24
 ineffective       0.54      0.65      0.59       232
 ingredients       0.84      0.34      0.48        47
   mandatory       0.68      0.53      0.59        89
        none       0.43      0.27      0.33        95
      pharma       0.50      0.65      0.56       141
   political       0.58      0.36      0.44        61
   religious       1.00      0.17      0.29         6
      rushed       0.61      0.50      0.55       159
 side-effect       0.64      0.81      0.72       405
 unnecessary       0.57      0.33      0.42        51

    accuracy                           0.59      1371
   macro avg       0.64      0.44      0.49      1371
weighted avg       0.59      0.59      0.57      1371


Results for Logistic Regression with C=4:
              precision    recall  f1-score   s

In [5]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer

import nltk
nltk.download('stopwords')
nltk.download('punkt')

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    text = re.sub(r'[^A-Za-z\s]', '', text)
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)
    tokens = [token for token in tokens if token not in stop_words]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(token) for token in tokens]
    preprocessed_text = ' '.join(tokens)

    return preprocessed_text

df['content_preprocessed'] = df['content'].apply(preprocess_text)
X_train, X_test, y_train, y_test = train_test_split(df['content_preprocessed'], df['concerns'], test_size=0.2, random_state=42)
tfidf_vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=4, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train.str.split())
y_test_encoded = mlb.transform(y_test.str.split())

for c_value in [1,2,3,4,5,6,7,8,9,10]:
    classifier = LogisticRegression(C=c_value, max_iter=1000)
    classifier.fit(X_train_tfidf, y_train_encoded.argmax(axis=1))
    y_pred = classifier.predict(X_test_tfidf)
    print(f"\nResults for Logistic Regression with C={c_value}:")
    print(classification_report(y_test_encoded.argmax(axis=1), y_pred, target_names=mlb.classes_))

[nltk_data] Downloading package stopwords to /home/aayush/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/aayush/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Results for Logistic Regression with C=1:
              precision    recall  f1-score   support

  conspiracy       0.70      0.23      0.35        61
     country       0.88      0.29      0.44        24
 ineffective       0.55      0.67      0.60       232
 ingredients       0.93      0.30      0.45        47
   mandatory       0.77      0.54      0.64        89
        none       0.53      0.18      0.27        95
      pharma       0.51      0.60      0.55       141
   political       0.78      0.23      0.35        61
   religious       0.00      0.00      0.00         6
      rushed       0.64      0.53      0.58       159
 side-effect       0.57      0.85      0.68       405
 unnecessary       0.70      0.31      0.43        51

    accuracy                           0.58      1371
   macro avg       0.63      0.39      0.45      1371
weighted avg       0.61      0.58      0.56      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=2:
              precision    recall  f1-score   support

  conspiracy       0.52      0.25      0.33        61
     country       0.60      0.38      0.46        24
 ineffective       0.56      0.65      0.60       232
 ingredients       0.79      0.32      0.45        47
   mandatory       0.66      0.56      0.61        89
        none       0.50      0.22      0.31        95
      pharma       0.51      0.62      0.56       141
   political       0.78      0.34      0.48        61
   religious       0.00      0.00      0.00         6
      rushed       0.62      0.52      0.57       159
 side-effect       0.61      0.83      0.70       405
 unnecessary       0.59      0.37      0.46        51

    accuracy                           0.59      1371
   macro avg       0.56      0.42      0.46      1371
weighted avg       0.59      0.59      0.57      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=3:
              precision    recall  f1-score   support

  conspiracy       0.56      0.31      0.40        61
     country       0.56      0.38      0.45        24
 ineffective       0.56      0.65      0.60       232
 ingredients       0.79      0.32      0.45        47
   mandatory       0.64      0.57      0.60        89
        none       0.46      0.24      0.32        95
      pharma       0.51      0.62      0.56       141
   political       0.69      0.39      0.50        61
   religious       0.00      0.00      0.00         6
      rushed       0.62      0.51      0.56       159
 side-effect       0.62      0.82      0.71       405
 unnecessary       0.54      0.37      0.44        51

    accuracy                           0.59      1371
   macro avg       0.55      0.43      0.47      1371
weighted avg       0.59      0.59      0.57      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=4:
              precision    recall  f1-score   support

  conspiracy       0.51      0.31      0.39        61
     country       0.62      0.42      0.50        24
 ineffective       0.55      0.63      0.59       232
 ingredients       0.81      0.36      0.50        47
   mandatory       0.62      0.56      0.59        89
        none       0.47      0.28      0.35        95
      pharma       0.51      0.61      0.56       141
   political       0.69      0.39      0.50        61
   religious       0.00      0.00      0.00         6
      rushed       0.60      0.50      0.55       159
 side-effect       0.64      0.81      0.71       405
 unnecessary       0.53      0.37      0.44        51

    accuracy                           0.59      1371
   macro avg       0.55      0.44      0.47      1371
weighted avg       0.59      0.59      0.58      1371



/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/aayush/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))



Results for Logistic Regression with C=5:
              precision    recall  f1-score   support

  conspiracy       0.50      0.33      0.40        61
     country       0.62      0.42      0.50        24
 ineffective       0.54      0.63      0.58       232
 ingredients       0.81      0.36      0.50        47
   mandatory       0.62      0.56      0.59        89
        none       0.47      0.29      0.36        95
      pharma       0.50      0.60      0.54       141
   political       0.67      0.39      0.49        61
   religious       1.00      0.17      0.29         6
      rushed       0.60      0.50      0.55       159
 side-effect       0.65      0.81      0.72       405
 unnecessary       0.51      0.37      0.43        51

    accuracy                           0.59      1371
   macro avg       0.63      0.45      0.50      1371
weighted avg       0.59      0.59      0.58      1371


Results for Logistic Regression with C=6:
              precision    recall  f1-score   s